In [ ]:
!pip install mrmr-selection optuna lightgbm xgboost imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import f_classif, chi2, mutual_info_classif
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from mrmr import mrmr_classif
import warnings
warnings.filterwarnings('ignore')

# =====================
# 1. VERİ OKUMA
# =====================
df = pd.read_csv("M_AT+_Dataset_All.csv", low_memory=False)
drop_cols = ["Id", "Index_2035", "Stage", "Abeta", "TAU", "PTAU"]
df = df.drop(columns=drop_cols)
df = df.select_dtypes(include=[np.number])

X = df.drop(columns=['Class'])
y = df['Class']
feature_names = X.columns.tolist()

TOP_N = 20  # Her yöntemden top 20 feature

print(f"Dataset: {X.shape}, Class dağılımı: {y.value_counts().to_dict()}")

# =====================
# 2. FEATURE SELECTION
# =====================
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)

minmax = MinMaxScaler()
X_minmax = pd.DataFrame(minmax.fit_transform(X), columns=feature_names)

selected_features = {}

# 1. LASSO
print("1. LASSO çalışıyor...")
from sklearn.linear_model import LassoCV
lasso = LassoCV(cv=3, random_state=42, max_iter=2000, n_jobs=-1)
lasso.fit(X_scaled, y)
coef = pd.Series(np.abs(lasso.coef_), index=feature_names)
selected_features['LASSO'] = coef[coef > 0].nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['LASSO'])} feature")

# 2. ANOVA
print("2. ANOVA çalışıyor...")
f_scores, _ = f_classif(X_scaled, y)
f_series = pd.Series(f_scores, index=feature_names)
selected_features['ANOVA'] = f_series.nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['ANOVA'])} feature")

# 3. Chi2
print("3. Chi2 çalışıyor...")
chi2_scores, _ = chi2(X_minmax, y)
chi2_series = pd.Series(chi2_scores, index=feature_names)
selected_features['Chi2'] = chi2_series.nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['Chi2'])} feature")

# 4. Mutual Information
print("4. Mutual Information çalışıyor...")
mi_scores = mutual_info_classif(X_scaled, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_names)
selected_features['MutualInfo'] = mi_series.nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['MutualInfo'])} feature")

# 5. Wilcoxon
print("5. Wilcoxon çalışıyor...")
class0 = X[y == 0]
class1 = X[y == 1]
wilcoxon_scores = {}
for col in feature_names:
    try:
        stat, p = wilcoxon(class0[col].values[:len(class1)],
                          class1[col].values[:len(class1)])
        wilcoxon_scores[col] = 1 - p
    except:
        wilcoxon_scores[col] = 0
wilcoxon_series = pd.Series(wilcoxon_scores)
selected_features['Wilcoxon'] = wilcoxon_series.nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['Wilcoxon'])} feature")

# 6. MRMR
print("6. MRMR çalışıyor...")
mrmr_features = mrmr_classif(X=X, y=y, K=TOP_N)
selected_features['MRMR'] = mrmr_features
print(f"   {len(selected_features['MRMR'])} feature")

# 7. Random Forest Importance
print("7. RF Importance çalışıyor...")
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_sel.fit(X_scaled, y)
rf_imp = pd.Series(rf_sel.feature_importances_, index=feature_names)
selected_features['RF_Importance'] = rf_imp.nlargest(TOP_N).index.tolist()
print(f"   {len(selected_features['RF_Importance'])} feature")

# 8. MRMR + LASSO (kesişim)
print("8. MRMR + LASSO kombinasyonu...")
mrmr_set = set(selected_features['MRMR'])
lasso_set = set(selected_features['LASSO'])
combined = list(mrmr_set.intersection(lasso_set))
if len(combined) < 5:
    combined = list(mrmr_set.union(lasso_set))[:TOP_N]
selected_features['MRMR+LASSO'] = combined[:TOP_N]
print(f"   {len(selected_features['MRMR+LASSO'])} feature")

# PCA (ayrı işlenir)
print("9. PCA çalışıyor...")
pca = PCA(n_components=TOP_N, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"   {TOP_N} component")

print("\nTüm feature selection tamamlandı!")

# =====================
# 3. CLASSIFIER TANIMLAMA
# =====================
classifiers = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM': SVC(probability=True, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(random_state=42, verbosity=-1),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000)
}

# =====================
# 4. YARDIMCI FONKSİYON
# =====================
def calculate_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'ROC_AUC': roc_auc_score(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Specificity': tn / (tn + fp),
        'Sensitivity': tp / (tp + fn)
    }

# =====================
# 5. CROSS VALIDATION
# =====================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

total = len(selected_features) * len(classifiers)
current = 0

for fs_name, features in selected_features.items():
    for clf_name, clf in classifiers.items():
        current += 1
        print(f"[{current}/{total}] {fs_name} + {clf_name}...")

        if fs_name == 'PCA':
            data_X = pd.DataFrame(X_pca)
            feat_list = [f"PC{i+1}" for i in range(TOP_N)]
        else:
            data_X = X[features].copy()
            feat_list = features

        data_y = y.copy()

        fold_metrics = {'ROC_AUC': [], 'Accuracy': [], 'Specificity': [], 'Sensitivity': []}

        for train_idx, test_idx in skf.split(data_X, data_y):
            X_train = data_X.iloc[train_idx]
            X_test = data_X.iloc[test_idx]
            y_train = data_y.iloc[train_idx]
            y_test = data_y.iloc[test_idx]

            # Scale
            sc = StandardScaler()
            X_train_sc = pd.DataFrame(sc.fit_transform(X_train), columns=X_train.columns)
            X_test_sc = pd.DataFrame(sc.transform(X_test), columns=X_test.columns)

            # SMOTE
            smote = SMOTE(random_state=42)
            X_res_arr, y_res = smote.fit_resample(X_train_sc, y_train)
            X_res = pd.DataFrame(X_res_arr, columns=X_train.columns)

            # Eğit
            clf.fit(X_res, y_res)
            y_pred = clf.predict(X_test_sc)
            y_proba = clf.predict_proba(X_test_sc)[:, 1]

            m = calculate_metrics(y_test, y_pred, y_proba)
            for k in fold_metrics:
                fold_metrics[k].append(m[k])

        results.append({
            'Feature_Selection': fs_name,
            'Classifier': clf_name,
            'ROC_AUC': round(np.mean(fold_metrics['ROC_AUC']), 3),
            'Accuracy': round(np.mean(fold_metrics['Accuracy']), 3),
            'Specificity': round(np.mean(fold_metrics['Specificity']), 3),
            'Sensitivity': round(np.mean(fold_metrics['Sensitivity']), 3),
            'Features_Used': ', '.join(feat_list[:5]) + '...' if len(feat_list) > 5 else ', '.join(feat_list)
        })
        print(f"   ROC AUC: {results[-1]['ROC_AUC']} | Sens: {results[-1]['Sensitivity']}")

# =====================
# 6. SONUÇLAR
# =====================
results_df = pd.DataFrame(results)

# Pivot tablo - ROC AUC
pivot_roc = results_df.pivot(index='Classifier', columns='Feature_Selection', values='ROC_AUC')
pivot_sens = results_df.pivot(index='Classifier', columns='Feature_Selection', values='Sensitivity')
pivot_spec = results_df.pivot(index='Classifier', columns='Feature_Selection', values='Specificity')
pivot_acc = results_df.pivot(index='Classifier', columns='Feature_Selection', values='Accuracy')

print("\n=== ROC AUC TABLOSU ===")
print(pivot_roc.to_string())

# Feature listeleri
features_detail = pd.DataFrame([
    {'Method': k, 'Feature_Count': len(v), 'Features': ', '.join(v)}
    for k, v in selected_features.items()
])

# Excel kaydet
with pd.ExcelWriter("AT_Full_Comparison.xlsx", engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='All Results', index=False)
    pivot_roc.to_excel(writer, sheet_name='ROC AUC Table')
    pivot_sens.to_excel(writer, sheet_name='Sensitivity Table')
    pivot_spec.to_excel(writer, sheet_name='Specificity Table')
    pivot_acc.to_excel(writer, sheet_name='Accuracy Table')
    features_detail.to_excel(writer, sheet_name='Selected Features', index=False)

print("\nExcel kaydedildi: AT_Full_Comparison.xlsx")

# Heatmap
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
import seaborn as sns

sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,0])
axes[0,0].set_title('ROC AUC')

sns.heatmap(pivot_sens, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0,1])
axes[0,1].set_title('Sensitivity')

sns.heatmap(pivot_spec, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1,0])
axes[1,0].set_title('Specificity')

sns.heatmap(pivot_acc, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1,1])
axes[1,1].set_title('Accuracy')

plt.suptitle('AT+ Classification - Feature Selection vs Classifier Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('AT_Comparison_Heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Grafik kaydedildi: AT_Comparison_Heatmap.png")